# drugMR: Interpreting Results

> *A guided walkthrough of a completed **drugMR** run outside the dashboard, dissecting the results tables directly, from the first genetic signal to the final evidence profile of each prioritised target.*

### Overview

Rather than browsing results in the dashboard, this notebook opens the raw results tables of a completed run and dissects them one stage at a time. Each section loads the relevant table, shows what its columns mean, and highlights what separates a convincing target from a weak one.

### Requirements

| | |
|:--|:--|
| **Repository** | The `drugMR` repository cloned locally. |
| **Environment** | Created with `env/bootstrap.sh`, which installs the `drugmr` package in editable mode. No separate `pip install` is needed. |
| **Kernel** | This notebook running on the environment's kernel (`.venv` in the repository root). |
| **Run** | A completed run under `runs/<run_id>/`. If the pipeline ran on a remote HPC, fetch it first with `notebooks/00_drugmr.ipynb`. |

In [ ]:
from drugmr.interpret import ResultsExplorer

In [ ]:
explorer = ResultsExplorer(
    run_id="AD_wu_csf_2026-10-09_18-51-14_1989e4d",
    root="../runs",
)

## Part 1: Run overview

> What this run is, the thresholds every later stage is judged against, and how many proteins survive each stage.

| Function | Returns |
|:--|:--|
| `run_info()` | Run identity, provenance, outcome GWAS inputs and every gate used |
| `funnel()` | Proteins remaining after each stage, with the share kept |

In [ ]:
explorer.run_info()

In [ ]:
explorer.funnel()

## Part 2: Causal estimates (cis-MR)

> The effect of each protein on the outcome, the instruments behind it, and whether the estimate holds across MR methods. A protein whose higher level raises risk is a target to inhibit; one whose higher level lowers risk is a target to activate.

| Function | Returns |
|:--|:--|
| `cis_mr()` | One row per protein: OR with 95% CI, P, FDR, effect and drug direction |
| `instruments()` | One row per instrument SNP: alleles, pQTL and outcome effects, F statistic, Steiger |
| `sensitivity()` | IVW, MR Egger and weighted median side by side (multi instrument proteins) |
| `forest_plot()` | Odds ratios with 95% CI for the prioritised targets |

In [ ]:
explorer.cis_mr() # proteins passing the cis-MR gate

In [ ]:
explorer.cis_mr(hits_only=False) # every protein tested, with a pass flag

In [ ]:
explorer.instruments()

In [ ]:
explorer.sensitivity()

In [ ]:
explorer.forest_plot();

## Part 3: Shared causal variant

> A causal estimate is only convincing if the protein and the outcome share the same causal variant. COLOC assumes one signal per trait; PWCoCo conditions on each independent signal, so it can recover windows with more than one.

| Function | Returns |
|:--|:--|
| `colocalisation()` | COLOC and PWCoCo per protein, the best PWCoCo pair and which method supports it |
| `coloc_priors()` | COLOC PP4 under each prior set, and whether it holds under all of them |
| `lead_variants()` | Top cis variant with outcome and pQTL effects on the same allele |

In [ ]:
explorer.colocalisation() # the prioritised set

In [ ]:
explorer.colocalisation(hits_only=False) # every cis-MR hit, colocalised or not

In [ ]:
explorer.coloc_priors()

In [ ]:
explorer.lead_variants()

## Part 4: xQTL triangulation

> Do molecular QTL of the same gene (xQTL) point to the same signal and direction as the pQTL? Bulk tissue and single cell xQTL are kept apart, and every check reports whether its direction agrees with cis-MR.

| Function | Returns |
|:--|:--|
| `smr()` | SMR and HEIDI per protein and tissue or cell type, with direction against cis-MR |
| `hyprcoloc()` | Clusters sharing one causal variant across pQTL, GWAS and xQTL |
| `pwcoco_qtl()` | PWCoCo of each xQTL against the pQTL and against the outcome GWAS |
| `directions()` | One row per target: does every line of evidence point the same way? |

In [ ]:
explorer.smr() # all significant SMR results

In [ ]:
explorer.smr(data_type="bulk") # bulk tissue xQTL only

In [ ]:
explorer.smr(data_type="single_cell") # single cell xQTL only

In [ ]:
explorer.hyprcoloc() # supporting clusters (pQTL + GWAS + xQTL, posterior >= 0.7)

In [ ]:
explorer.hyprcoloc(supported_only=False) # every cluster, with a support flag

In [ ]:
explorer.pwcoco_qtl()

In [ ]:
explorer.directions()

## Part 5: Safety and repurposing

> Other traits the same instruments affect. A trait moving in the same direction as the outcome is a possible additional indication; the opposite direction is a possible adverse effect.

| Function | Returns |
|:--|:--|
| `phewas()` | Bonferroni significant traits per protein (FinnGen, then UK Biobank), with their class |

In [ ]:
explorer.phewas()

## Part 6: Evidence summary

> One row per prioritised target, every stage summarised.

| Function | Returns |
|:--|:--|
| `targets()` | cis-MR, colocalisation, xQTL support, directions and PheWAS in a single table |

In [ ]:
explorer.targets()

## Part 7: A single target, end to end (TREM2)

> Every function above also takes a protein, by symbol (any case) or full protein ID, for example `explorer.smr("TREM2")`. `target()` runs them all for one protein in reading order, and `regional_plot()` shows its cis window.

| Function | Returns |
|:--|:--|
| `target(protein)` | Every table above for one protein, as a dictionary of tables |
| `regional_plot(protein)` | Outcome GWAS and pQTL across the cis window, coloured by LD with the lead variant |

In [ ]:
trem2 = explorer.target("TREM2")

In [ ]:
explorer.regional_plot("TREM2");